In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler
import os
import joblib
import warnings
warnings.filterwarnings("ignore")

# ========================= CONFIG =========================
CHANNELS = list(range(41, 47))          # channels 41 to 46
WINDOW_SIZE = 60
BATCH_SIZE = 64
EPOCHS = 25
UNITS = 80

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("=== LSTM Reconstruction Baseline using TensorFlow/Keras ===\n")

# ========================= 1. PREPROCESS TRAIN (cached) =========================
cache_file = os.path.join(CACHE_DIR, "preprocessed_lstm_train_channels41-46.pkl")

if os.path.exists(cache_file):
    print("Loading cached train data...")
    data_scaled, df_train, scaler = joblib.load(cache_file)
else:
    print("Preprocessing train.parquet...")
    df_train = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/train.parquet")
    
    channel_cols = [f'channel_{i}' for i in CHANNELS]
    df_train = df_train[['id'] + channel_cols + ['is_anomaly']].copy()
    df_train = df_train.sort_values('id').reset_index(drop=True)
    
    df_train[channel_cols] = df_train[channel_cols].ffill().bfill()
    
    data = df_train[channel_cols].values
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    
    joblib.dump((data_scaled, df_train, scaler), cache_file)
    print(f"Train data cached. Shape: {data_scaled.shape}")

# ========================= 2. CREATE SEQUENCES =========================
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(data_scaled, WINDOW_SIZE)
print(f"Training sequences created → X shape: {X_train.shape}")

# ========================= 3. BUILD LSTM MODEL =========================
model = Sequential([
    LSTM(UNITS, input_shape=(WINDOW_SIZE, len(CHANNELS)), return_sequences=False),
    Dropout(0.3),
    Dense(len(CHANNELS))
])

model.compile(optimizer='adam', loss='mse')
model.summary()

# ========================= 4. TRAINING =========================
model_path = os.path.join(CACHE_DIR, f"lstm_tf_model_channels41-46_ep{EPOCHS}.h5")

if os.path.exists(model_path):
    print("\nLoading trained LSTM model from cache...")
    model = tf.keras.models.load_model(model_path)
    print("Model loaded successfully!")
else:
    print("\nTraining LSTM model...")
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        verbose=1,
        shuffle=True
    )
    model.save(model_path)
    print("Training completed and model saved!")

# ========================= 5. INFERENCE ON TEST =========================
print("\nLoading test.parquet for inference...")
df_test = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/test.parquet")

channel_cols = [f'channel_{i}' for i in CHANNELS]
df_test = df_test[['id'] + channel_cols].copy()
df_test = df_test.sort_values('id').reset_index(drop=True)
df_test[channel_cols] = df_test[channel_cols].ffill().bfill()

test_data = scaler.transform(df_test[channel_cols].values)

X_test, _ = create_sequences(test_data, WINDOW_SIZE)

print("Computing reconstruction errors...")
predictions = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

# Reconstruction error (MSE)
reconstruction_errors = np.mean((predictions - X_test[:, -1, :]) ** 2, axis=1)

# Threshold
threshold = np.percentile(reconstruction_errors, 97.5)
anomaly_flag = (reconstruction_errors > threshold).astype(int)

print(f"Threshold used: {threshold:.6f}")
print(f"Predicted anomalies in test: {anomaly_flag.sum()} ({anomaly_flag.mean():.4%})")

# ========================= 6. CREATE SUBMISSION =========================
submission = pd.DataFrame({
    'id': df_test['id'].iloc[WINDOW_SIZE:].reset_index(drop=True),
    'anomaly_flag': anomaly_flag
})

submission.to_parquet("submission_lstm_tf_baseline.parquet", index=False)

print("\n✅ Submission saved as 'submission_lstm_tf_baseline.parquet'")
print("Upload this file to Kaggle as Late Submission to get the real corrected event-wise F0.5 score.")

2026-04-21 09:50:46.162872: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


=== LSTM Reconstruction Baseline using TensorFlow/Keras ===

Loading cached train data...
Training sequences created → X shape: (14728261, 60, 6)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 80)             │        27,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           486 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,326 (110.65 KB)

 Trainable params: 28,326 (110.65 KB)

 Non-trainable params: 0 (0.00 B)


Training LSTM model...
Epoch 1/25
 36865/207117 ━━━━━━━━━━━━━━━━━━━━ 6:32:12 138ms/step - loss: 0.0941